In [1]:
import re

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\diego\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\diego\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
from pathlib import Path

def load_imdb_data(path):
    path = Path(path)
    data = []
    labels = []

    for label in ['pos', 'neg']:
        dir_path = path / label

        for file_path in sorted(dir_path.glob('*.txt')):
            with open(file_path, 'r', encoding='utf-8') as file:
                data.append(file.read())
                labels.append(1 if label == 'pos' else 0)

    return pd.DataFrame({'review': data, 'sentiment': labels})


dataset_root = Path('data/aclImdb_v1/aclImdb')

train_df = load_imdb_data(dataset_root / 'train')
test_df = load_imdb_data(dataset_root / 'test')

In [4]:
english_stopwords = stopwords.words('english')

neg_words = [
    'no', 'nor', 'not', 'ain', 'aren', "aren't", 'don', "don't",
    'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't",
    'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't",
    'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't",
    'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't",
    'wasn', "wasn't", 'weren', "weren't", 'won', "won't",
    'wouldn', "wouldn't"
]

english_stopwords = [w for w in english_stopwords if w not in neg_words]


def normalize_text(text):
    text = re.sub(r'<.*?>', ' ', text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def preprocess_stem(text):
    text = text.lower()
    stemmed_tokens = [stemmer.stem(token) for token in text.split()]
    return normalize_text(' '.join(stemmed_tokens))


def preprocess_lemma(text):
    text = text.lower()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in text.split()]
    return normalize_text(' '.join(lemmatized_tokens))


def build_stopwords(preprocess_fn):
    processed_stopwords = {preprocess_fn(word) for word in english_stopwords}
    return sorted(word for word in processed_stopwords if word)

In [6]:
RANDOM_STATE = 42

train_exp_df = train_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_exp_df = test_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

In [7]:
def run_experiment(vectorizer, preprocess_fn=None):
    train_text = train_exp_df['review']
    test_text = test_exp_df['review']

    if preprocess_fn is not None:
        train_text = train_text.apply(preprocess_fn)
        test_text = test_text.apply(preprocess_fn)

    x_train_vec = vectorizer.fit_transform(train_text)
    y_train = train_exp_df['sentiment']

    x_test_vec = vectorizer.transform(test_text)
    y_test = test_exp_df['sentiment']

    model = LogisticRegression(
        solver='liblinear',
        max_iter=1000,
        random_state=RANDOM_STATE
    )

    model.fit(x_train_vec, y_train)

    y_train_pred = model.predict(x_train_vec)
    y_test_pred = model.predict(x_test_vec)

    metrics = {
        'accuracy_train': accuracy_score(y_train, y_train_pred),
        'accuracy_test': accuracy_score(y_test, y_test_pred),
        'precision_train': precision_score(y_train, y_train_pred),
        'precision_test': precision_score(y_test, y_test_pred),
        'recall_train': recall_score(y_train, y_train_pred),
        'recall_test': recall_score(y_test, y_test_pred),
        'f1_train': f1_score(y_train, y_train_pred),
        'f1_test': f1_score(y_test, y_test_pred),
    }

    return model, vectorizer, metrics

In [8]:
stem_stopwords = build_stopwords(preprocess_stem)
lemma_stopwords = build_stopwords(preprocess_lemma)

experiments = [
    ('CountVectorizer-stem', preprocess_stem, CountVectorizer(stop_words=stem_stopwords, lowercase=False)),
    ('CountVectorizer-lemma', preprocess_lemma, CountVectorizer(stop_words=lemma_stopwords, lowercase=False)),
    ('TfidfVectorizer-stem', preprocess_stem, TfidfVectorizer(stop_words=stem_stopwords, lowercase=False)),
    ('TfidfVectorizer-lemma', preprocess_lemma, TfidfVectorizer(stop_words=lemma_stopwords, lowercase=False)),
]

results = []
model_runs = []

for exp_name, preprocess_fn, vectorizer in experiments:
    model, fitted_vectorizer, metrics = run_experiment(vectorizer, preprocess_fn=preprocess_fn)

    results.append({
        'vectorizer': exp_name,
        **metrics,
    })

    model_runs.append({
        'name': exp_name,
        'model': model,
        'fitted_vectorizer': fitted_vectorizer,
    })

results_df = pd.DataFrame(results).sort_values('accuracy_test', ascending=False)
results_df

,vectorizer,accuracy_train,accuracy_test,precision_train,precision_test,recall_train,recall_test,f1_train,f1_test
3,TfidfVectorizer-lemma,0.93756,0.88156,0.931043,0.882877,0.94512,0.87984,0.938029,0.881356
2,TfidfVectorizer-stem,0.93112,0.87688,0.923464,0.876639,0.94016,0.87720,0.931737,0.876919
1,CountVectorizer-lemma,0.99784,0.86016,0.997999,0.866314,0.99768,0.85176,0.997840,0.858975
0,CountVectorizer-stem,0.99744,0.85740,0.997520,0.861419,0.99736,0.85184,0.997440,0.856603


In [9]:
def get_top_features(model, fitted_vectorizer, top_n=10):
    feature_names = fitted_vectorizer.get_feature_names_out()
    coefs = model.coef_[0]

    top_pos_idx = coefs.argsort()[::-1][:top_n]
    top_neg_idx = coefs.argsort()[:top_n]

    top_positive = [(feature_names[idx], coefs[idx]) for idx in top_pos_idx]
    top_negative = [(feature_names[idx], coefs[idx]) for idx in top_neg_idx]

    return top_positive, top_negative


def print_compact_feature_table(model_runs, sentiment='positive', top_n=10):
    table_data = {}

    for run in model_runs:
        top_positive, top_negative = get_top_features(
            run['model'],
            run['fitted_vectorizer'],
            top_n=top_n
        )

        selected = top_positive if sentiment == 'positive' else top_negative
        table_data[run['name']] = [f"{word} ({coef:.3f})" for word, coef in selected]

    compact_df = pd.DataFrame(table_data, index=[f"#{i}" for i in range(1, top_n + 1)])
    compact_df.index.name = 'rank'

    display(compact_df)


print('Top 10 positive words across experiments')
print_compact_feature_table(model_runs, sentiment='positive', top_n=10)

print('\nTop 10 negative words across experiments')
print_compact_feature_table(model_runs, sentiment='negative', top_n=10)

Top 10 positive words across experiments


,CountVectorizer-stem,CountVectorizer-lemma,TfidfVectorizer-stem,TfidfVectorizer-lemma
rank,,,,
#1,excellent (1.701),refreshing (1.597),great (7.192),great (7.066)
#2,squirrel (1.471),wonderfully (1.572),excel (5.423),excellent (6.150)
#3,units (1.418),funniest (1.398),love (5.160),best (5.010)
#4,flawless (1.415),flawless (1.351),perfect (4.988),perfect (4.889)
#5,surprisingli (1.389),superb (1.317),best (4.964),wonderful (4.624)
#6,superb (1.333),excellent (1.288),enjoy (4.914),well (4.409)
#7,witty (1.332),perfect (1.280),well (4.558),favorite (4.262)
#8,carrey (1.271),appreciated (1.262),favorit (4.287),amazing (4.223)
#9,refresh (1.260),units (1.261),fun (4.007),loved (3.942)



Top 10 negative words across experiments


,CountVectorizer-stem,CountVectorizer-lemma,TfidfVectorizer-stem,TfidfVectorizer-lemma
rank,,,,
#1,awful (-2.131),worst (-2.150),worst (-9.258),worst (-9.179)
#2,worst (-2.049),disappointment (-2.144),bad (-7.364),bad (-7.335)
#3,poorli (-1.913),waste (-2.014),wast (-7.248),awful (-6.551)
#4,disappointment (-1.703),poorly (-1.794),poor (-5.411),waste (-6.434)
#5,wast (-1.676),awful (-1.697),bore (-5.310),boring (-5.719)
#6,terrible (-1.491),disappointing (-1.548),noth (-5.084),poor (-5.329)
#7,alright (-1.434),unfunny (-1.535),awful (-4.824),terrible (-4.751)
#8,lacking (-1.358),alright (-1.508),aw (-4.619),nothing (-4.736)
#9,badli (-1.349),laughable (-1.476),no (-4.605),worse (-4.700)


In [10]:
best_model_name = results_df.iloc[0]['vectorizer']
best_run = next(run for run in model_runs if run['name'] == best_model_name)

best_model = best_run['model']
best_vectorizer = best_run['fitted_vectorizer']

print("Best model:", best_model_name)
print("Best test accuracy:", results_df.iloc[0]['accuracy_test'])

Best model: TfidfVectorizer-lemma
Best test accuracy: 0.88156


In [11]:
my_review = "The movie was entertaining and well made. The story was interesting, the acting was strong, and I enjoyed watching it."

if 'lemma' in best_model_name:
    processed_review = preprocess_lemma(my_review)
else:
    processed_review = preprocess_stem(my_review)

review_vec = best_vectorizer.transform([processed_review])
prediction = best_model.predict(review_vec)[0]

print("Prediction:", "positive" if prediction == 1 else "negative")

Prediction: positive


### Which vectorizer and preprocessing combination gives the best performance?
El mejor modelo fue **TfidfVectorizer con lemmatization**. Despues de reordenar el pipeline para aplicar stemming o lemmatization antes de la normalizacion final, TF-IDF siguio siendo la mejor opcion en el conjunto de prueba.

### Do you see any interesting differences in the top features?
En las top features se nota que TF-IDF usa palabras más claras para el sentimiento, como **great**, **excellent**, **best**, **worst**, **bad** y **awful**. En cambio, stemming corta las palabras y genera términos menos legibles como **excel**, **wast**, **aw** o **favorit**. Lemmatization mantiene palabras más naturales y fáciles de interpretar.

### Do you agree with the model's prediction?

El modelo predijo un sentimiento positivo. Estoy de acuerdo con la predicción porque la reseña usaba palabras positivas.

### Write your final remarks.

En conclusion, esta actividad muestra que el preprocesamiento si afecta el desempeno del modelo. Tambien deja claro que el orden del pipeline importa: en esta version primero se aplica stemming o lemmatization y despues se normaliza el texto para cumplir con la indicacion vista en clase.